In [2]:
pip install -q langgraph langchain-openai pydantic

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

# 1. Dynamically find the absolute path to your .env file
# (This fixes the pathing issue in Jupyter Notebooks/VS Code)
env_path = Path('.') / '.env'

# 2. Explicitly load the .env file from that path
load_dotenv(dotenv_path=env_path)

# 3. Look up the variable safely
if "GITHUB_TOKEN" not in os.environ:
    raise ValueError("❌ GITHUB_TOKEN environment variable is missing! Please configure it.")

print("✅ Token loaded successfully from system environment.")

✅ Token loaded successfully from system environment.


In [5]:
# ==========================================
# CELL 3: WEEK 1 SYSTEM PROMPTS & TEMPLATES
# ==========================================

WRITER_SYSTEM_PROMPT = """You are a premier Product Management Writer Agent. 
Your goal is to transform messy meeting notes into a deeply structured, polished Product Requirement Document (PRD) in Markdown format.

Every PRD you generate MUST contain these 6 exact sections:
1. Title
2. Executive Summary
3. User Stories
4. Technical Specifications
5. Edge Cases
6. Success  Metrics
----
---
FEW-SHOT EXAMPLE OF DESIRED PRD OUTPUT STRUCTURE:
# PRD: Core Checkout Refactor
## 1. Executive Summary
Streamlining our application's final billing operations to implement reliable, rapid transactions.

## 2. User Stories
* As a shopper, I want to complete my purchase using one-click checkout so that I spend less time typing payment credentials.

## 3. Technical Specifications
* Backend Framework: Node.js 
* Integration Point: Stripe API v3 Gateway
* Target Endpoint: `POST /api/v1/checkout/charge`

## 4. Edge Cases
* The user loses data connectivity or signals fluctuate mid-transit during card validation.
* The transaction is declined by the processor due to insufficient funds.

## 5. Success Metrics
* The core API latency response remains strictly beneath 200ms.
* Zero dropped transactional states across edge conditions.
---
"""

CRITIC_SYSTEM_PROMPT = """You are a strict QA Compliance Reviewer Agent. 
Your task is to review the provided PRD draft against our strict company template rubric.

You must explicitly evaluate the document for:
- Any missing sections out of the required 6 (Title, Summary, User Stories, Tech Specs, Edge Cases, Success Metrics).
- Low quality or ambiguous details.

You must output a structured evaluation containing an assessment score from 0 to 100, a clear list of what elements need fixing, and a boolean true/false flag indicating passing status."""

print("✅ Cell 3: Week 1 System Prompts loaded successfully.")

✅ Cell 3: Week 1 System Prompts loaded successfully.


In [ ]:
# ==========================================
# CELL 4: WEEK 2 DATA SCHEMAS & AGENT NODES
# ==========================================

from typing import Dict, Any
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# 1. State Management Design
class AgentWorkflowState(BaseModel):
    initial_input: str = ""
    current_draft: str = ""
    critic_feedback: str = ""
    critic_score: int = 0
    revision_count: int = 0
    max_revisions: int = 2

# 2. Structured Output Schema for Critic Validation
class CriticEvaluation(BaseModel):
    score: int = Field(description="Rubric score from 0-100.")
    feedback: str = Field(description="Detailed notes on missing items or parts requiring revision.")
    is_passing: bool = Field(description="True if score is 80 or higher, otherwise False.")

# 3. Model Configuration via Free GitHub Inference Endpoint
llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com",
    temperature=0.2
)

# 4. Define the Writer Agent Node Function
def writer_agent(state: AgentWorkflowState) -> Dict[str, Any]:
    print(f"\n[Writer Agent] Constructing/Amending Draft (Revision Counter: {state.revision_count + 1})...")
    
    if not state.current_draft:
        user_content = f"Draft an extensive structural PRD from these base inputs:\n\n{state.initial_input}"
    else:
        user_content = f"Current Draft:\n{state.current_draft}\n\nStrict Critique to Implement:\n{state.critic_feedback}"
        
    messages = [
        SystemMessage(content=WRITER_SYSTEM_PROMPT), 
        HumanMessage(content=user_content)
    ]
    response = llm.invoke(messages)
    return {"current_draft": response.content, "revision_count": state.revision_count + 1}

# 5. Define the Critic Agent Node Function
def critic_agent(state: AgentWorkflowState) -> Dict[str, Any]:
    print("[Critic Agent] Auditing document metrics and structure compliance...")
    
    # Enforce structured output from gpt-4o using the Pydantic schema
    structured_llm = llm.with_structured_output(CriticEvaluation)
    messages = [
        SystemMessage(content=CRITIC_SYSTEM_PROMPT), 
        HumanMessage(content=state.current_draft)
    ]
    evaluation = structured_llm.invoke(messages)
    
    print(f"[Critic Agent] Audit Result -> Score: {evaluation.score}/100 | Approved: {evaluation.is_passing}")
    return {"critic_feedback": evaluation.feedback, ""
    "critic_score": evaluation.score}

print("✅ Cell 4: Week 2 State Schemas and Agent Nodes compiled successfully.")
#----

✅ Cell 4: Week 2 State Schemas and Agent Nodes compiled successfully.


In [ ]:
# ==========================================
# CELL 5: LANGGRAPH COMPILATION & EXECUTION
# ==========================================

from typing import Literal
from langgraph.graph import StateGraph, END

# 1. Define Routing Logic (Circuit Breaker Edge)
def routing_logic(state: AgentWorkflowState) -> Literal["writer", "end"]:
    # If the critic gives a passing grade (>= 80) OR we hit the max revisions, stop.
    if state.critic_score >= 80 or state.revision_count >= state.max_revisions:
        print("[Router] Graph conditions met. Routing to completion...")
        return "end"
    
    # Otherwise, loop back to the writer for adjustments
    print(f"[Router] Revision needed (Current Score: {state.critic_score}/100). Routing back to Writer...")
    return "writer"

# 2. Build the Graph Workflow Structure
workflow = StateGraph(AgentWorkflowState)

# Add our active processing nodes
workflow.add_node("writer", writer_agent)
workflow.add_node("critic", critic_agent)

# Configure the processing sequence
workflow.set_entry_point("writer")
workflow.add_edge("writer", "critic")

# Add the conditional routing loop edge out of the critic node
workflow.add_conditional_edges(
    "critic", 
    routing_logic, 
    {
        "writer": "writer", 
        "end": END
    }
)

# 3. Compile the Executable Application Graph
app = workflow.compile()

# 4. TEST EXECUTION: Run a messy setup to test the graph validation loop
test_notes = (
    "App Name: EasySplit. An application allowing users to scan receipt images "
    "and instantly split totals with friends using OCR. Built with Python and FastAPI."
)

print("🚀 Starting Multi-Agent Review Graph Pipeline...")
initial_state = AgentWorkflowState(initial_input=test_notes)
final_output = app.invoke(initial_state)

print("\n==============================================")
print("🏆 FINAL POLISHED COMPLIANT PRD GENERATED:")
print("==============================================")
print(final_output["current_draft"])
#-----